# DiffusionGemma Workshop

<a target="_blank" href="https://colab.research.google.com/github/ro-witthawin/ro-witthawin-workshop/blob/main/workshops/DiffusionGemma/DiffusionGemma.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>

This notebook runs `google/diffusiongemma-26B-A4B-it` with Hugging Face Transformers and shows how a diffusion language model refines generated text over multiple draft steps.

DiffusionGemma is different from a standard left-to-right language model. Instead of committing one token at a time, it works over a generation canvas and iteratively denoises token blocks until the answer stabilizes. The streamer section below makes that refinement process visible.

## Before You Run

Install the workshop dependencies from this folder:

```bash
pip install -r requirements.txt
```

For Colab, install the same core packages in a setup cell if needed:

```python
%pip install -U torch "transformers>=5.11.0" accelerate
```

The full model is large. Use a GPU runtime with enough memory; if you are only testing notebook flow, reduce `max_new_tokens` in the generation cell.

## Notebook Flow

1. Load the processor and block-diffusion model.
2. Write a chat-style prompt.
3. Convert the prompt into model-ready tensors.
4. Create a live streamer for draft diffusion steps.
5. Generate text and watch the drafts evolve.
6. Decode and render the final response.

## 1. Load The Model And Processor

`AutoProcessor` loads the tokenizer, chat template, and model-specific preprocessing logic from the Hugging Face model repo. `DiffusionGemmaForBlockDiffusion` loads the model class that implements DiffusionGemma's block-diffusion generation behavior.

`dtype="auto"` lets Transformers choose the model's preferred tensor type for the current hardware. `device_map="auto"` asks Accelerate to place model weights on available devices automatically, which is useful for large checkpoints.

In [1]:
from transformers import DiffusionGemmaForBlockDiffusion, AutoProcessor

MODEL_ID = "google/diffusiongemma-26B-A4B-it"

# Load model
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = DiffusionGemmaForBlockDiffusion.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto",
)

/home/chula_ai_spark001/miniconda3/envs/vlm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 11 files: 100%|█████████████████████████████████████████████████████| 11/11 [00:00<00:00, 90465.38it/s]
/home/chula_ai_spark001/miniconda3/envs/vlm/lib/python3.12/site-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  warnings.warn(
Loading weights: 100%|██████████████████████████████████████████████████████| 1047/1047 [05:56<00:00,  2.93it/s]


## 2. Define The Prompt

The prompt uses a chat-message format with a `role` and `content`. This lets the processor apply Gemma's chat template correctly and makes it easy to extend the notebook later with system messages or multi-turn conversations.

Change the `content` string to test different workshop questions.

In [20]:
# Prompt
message = [
    {"role": "user", "content": "What is diffusion llm?"}
]

## 3. Convert The Prompt Into Model Inputs

`apply_chat_template` formats the message using the model's expected conversation template, tokenizes it, and returns PyTorch tensors.

`add_generation_prompt=True` appends the model-turn marker so generation starts in the right place. The final `.to(model.device)` moves the tensors onto the same device as the model before inference.

In [21]:
# Process input
input_ids = processor.apply_chat_template(
    message,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

## 4. Stream The Diffusion Drafts

`TextDiffusionStreamer` exposes intermediate text drafts while the model denoises its generation canvas. The custom `LiveDiffusionStreamer` displays each draft as Markdown so you can see how the answer changes from step to step.

`put_draft` handles intermediate drafts. `put` handles the final batch that the model commits after diffusion sampling finishes.

In [ ]:
import sys
from IPython.display import display, Markdown
from transformers import TextDiffusionStreamer

class LiveDiffusionStreamer(TextDiffusionStreamer):
    def __init__(self, tokenizer):
        super().__init__(tokenizer)
        self.last_lines = 0
        self.step = 0

    def put_draft(self, value=None, **kwargs):
        text = self.tokenizer.decode(
            value,
            skip_special_tokens=True,
        )

        if self.last_lines:
            sys.stdout.write(f"\033[{self.last_lines}F")
        line = "=" * 20 + "\n\n"
        output = f"{line}# Step {self.step}\n\n{text[0]}\n\n"
        display(Markdown(output))
        # print(processor.decode(value)[0])
        # sys.stdout.flush()
        self.last_lines = output.count("\n")
        self.step += 1

    def put(self, value=None, **kwargs):
        text = self.tokenizer.decode(
            value,
            skip_special_tokens=True,
        )

        if self.last_lines:
            sys.stdout.write(f"\033[{self.last_lines}F")

        output = f"\n\n# Final Batch\n\n{text[0]}\n\n"
        display(Markdown(output))
        # sys.stdout.flush()

## 5. Generate The Response

This cell creates the streamer and calls `model.generate`. The `streamer` argument sends intermediate diffusion states to `LiveDiffusionStreamer`, while `max_new_tokens` controls the maximum length of generated output.

For faster workshop demos or smaller GPU runtimes, start with a lower value such as `512` and increase it after the notebook is working.

In [76]:
streamer = LiveDiffusionStreamer(tokenizer=processor.tokenizer)

output = model.generate(
    **input_ids,
    max_new_tokens=4096,
    streamer=streamer,
)



# Final Batch

user
What is diffusion llm?
model




====================

# Step 0

thought
A **Diffusion LLM** refers to a of of of that that that that ** ** ** Models**** ( ( Models ( ( ( ( ( ( ( ( ( ( ( (,,,,,,,,,,,,,,,





























.













.







...... ( ( (. ( (. ( (..,..........................................********************** ( are used in image ( ( (.............................................................................................



====================

# Step 1

thought
A **Diffusion LLM** is a emerging class of Large Language Language that applies ** **Diffusion**** (used in like like like like Stable---E)))) the **** ( ( ( ( (







 understand,,,,,,,, to to to to how how how how













........ ( ( ( ( ( ( ( ( ( (,,,,,,, ** ** **gressive They They They They They They They They word word word the the the the the the the the the the the the the the........********************** the is used to the................................................................................................



====================

# Step 2

thought
A **Diffusion LLM** is an emerging type of Large Language model that uses **Diffusion Models** (the technology used in generators like like Stablejourney or D--E) of the the ** ** ** architecture** ( ( ( like like).).

 understand understand LL LL LL,,,, to to to how how how how how..









.... The The: Standard (


 ( ( ( ( ( GPT GPT GPT GPT ** ** ** ****. They text text text text one time time time time time They the the the the the the the the the the the the the the the the the******      *   *   *      .. the the the the the the the the the the the the the the the the the the the the the the the the the the the............... Diffusion Diffusion

 Diffusion
 Diffusion Diffusion

 Diffusion
 Diffusion Diffusion........... a image a image image image image image image image a image a image image a......... a... a.......



====================

# Step 3

thought
A **Diffusion LLM** is an emerging type of large language model that uses **diffusion models** (the technology behind image generators generators Midjourneyjourney or D--)) of the the ** **Transformergressive** architecture used by GPT-4.

To understand what Diffusion LLM, it helps to compare how how how how models work.







 1.. The Way:: Aut ( ( ()

 LL ( (like-4) are ** **ore**.**. They text text one token at ( ( () right right right
**         :**:**:** at the the the the the the the the the the word the word word the the word word word....**                  :**:** the the the the the the the the the the the the the the the the the the the the the the the the the the the the...................



.







 image image
 image image image image...........................



...

====================

# Step 50

 sentence with a beginning and an end, it can "diffuse" the middle to fit both ends perfectly. Autoregressive models struggle with this because they can only move forward.

### The Challenges (Why aren't we using them yet?)
Despite the potential, Diffusion LLMs are currently in the research phase:
1.  **Discrete vs. Continuous:** Diffusion works best on continuous data (like pixels in an image). Text is "discrete"—words are distinct units. It is much harder to "slightly blur" a word than it is a color. 
2.  **Computational Cost:** While they can be parallel, each "step" of the diffusion process is mathematically heavy. If the model needs 50 steps to refine a sentence, it might still be slower than just predicting the words.

### Summary
A **Diffusion LLM** is an attempt to move away from "one-word-at-a-time" generation toward "all-at-once" refinement. While not yet the industry standard, it is a major area of AI research aimed at making AI faster and better capable of handling complex, long-form structures.



====================

# Step 51

 sentence with a beginning and an end, it can "diffuse" the middle to fit both ends perfectly. Autoregressive models struggle with this because they can only move forward.

### The Challenges (Why aren't we using them yet?)
Despite the potential, Diffusion LLMs are currently in the research phase because
1.  **Discrete vs. Continuous:** Diffusion works best on continuous data (like pixels in an image). Text is "discrete"—words are distinct units. It is much harder to "slightly blur" a word than it is a color. 
2.  **Computational Cost:** While they can be parallel, each "step" of the diffusion process is mathematically heavy. If the model needs 50 steps to refine a sentence, it might still be slower than just predicting the words.

### Summary
A **Diffusion LLM** is an attempt to move away from "one-word-at-a-time" generation toward "all-at-once" refinement. While not yet the industry standard, it is a major area of AI research aimed at making AI faster and better capable of handling complex, long-form structures.



====================

# Step 52

 sentence with a beginning and an end, it can "diffuse" the middle to fit both ends perfectly. Autoregressive models struggle with this because they can only move forward.

### The Challenges (Why aren't we using them yet?)
Despite the potential, Diffusion LLMs are currently in the research phase because
1.  **Discrete vs. Continuous:** Diffusion works best on continuous data (like pixels in an image). Text is "discrete"—words are distinct units. It is much harder to "slightly blur" a word than it is a color. 
2.  **Computational Cost:** While they can be parallel, each "step" of the diffusion process is mathematically heavy. If the model needs 50 steps to refine a sentence, it might still be slower than just predicting the words.

### Summary
A **Diffusion LLM** is an attempt to move away from "one-word-at-a-time" generation toward "all-at-once" refinement. While not yet the industry standard, it is a major area of AI research aimed at making AI faster and more capable of handling complex, long-form structures.



====================

# Step 53

 sentence with a beginning and an end, it can "diffuse" the middle to fit both ends perfectly. Autoregressive models struggle with this because they can only move forward.

### The Challenges (Why aren't we using them yet?)
Despite the potential, Diffusion LLMs are currently in the research phase because
1.  **Discrete vs. Continuous:** Diffusion works best on continuous data (like pixels in an image). Text is "discrete"—words are distinct units. It is much harder to "slightly blur" a word than it is a color. 
2.  **Computational Cost:** While they can be parallel, each "step" of the diffusion process is mathematically heavy. If the model needs 50 steps to refine a sentence, it might still be slower than just predicting the words.

### Summary
A **Diffusion LLM** is an attempt to move away from "one-word-at-a-time" generation toward "all-at-once" refinement. While not yet the industry standard, it is a major area of AI research aimed at making AI faster and more capable of handling complex, long-form structures.





# Final Batch

 sentence with a beginning and an end, it can "diffuse" the middle to fit both ends perfectly. Autoregressive models struggle with this because they can only move forward.

### The Challenges (Why aren't we using them yet?)
Despite the potential, Diffusion LLMs are currently in the research phase because
1.  **Discrete vs. Continuous:** Diffusion works best on continuous data (like pixels in an image). Text is "discrete"—words are distinct units. It is much harder to "slightly blur" a word than it is a color. 
2.  **Computational Cost:** While they can be parallel, each "step" of the diffusion process is mathematically heavy. If the model needs 50 steps to refine a sentence, it might still be slower than just predicting the words.

### Summary
A **Diffusion LLM** is an attempt to move away from "one-word-at-a-time" generation toward "all-at-once" refinement. While not yet the industry standard, it is a major area of AI research aimed at making AI faster and more capable of handling complex, long-form structures.



## 6. Inspect The Raw Decoded Output

This quick decode is useful for debugging because it lets you inspect exactly what the model returned. Depending on decode options, you may see special tokens, channel markers, padding, or other template artifacts that are normally hidden from end users.

In [62]:
processor.decode(output[0])[0]

'<bos><|turn>user\nWhat is diffusion llm?<turn|>\n<|turn>model\n<|channel>thought\n<channel|>A **Diffusion LLM** is an emerging type of large language model that applies **diffusion processes**—the technology behind AI image generators like Midjourney and DALL-E—to text generation instead of the traditional "autoregressive" method used by GPT.\n\nTo understand Diffusion LLMs, it helps to compare them to how ChatGPT actually works today.\n\n---\n\n### 1. The Traditional Way: Autoregressive Models\nMost current LLMs (like GPT-4 or Llama) are **autoregressive**. They generate text one token at a time, from left to right.\n* **How it works:** The model looks at the prompt, predicts the next word, adds that word to the sequence, and predicts the next one.\n* **The Limitation:** This is sequential. If the model makes a mistake early in a sentence, the entire following structure often drifts off because it is building on its own error.\n\n### 2. The Diffusion Way: Diffusion LLMs\nDiffusion mo

## 7. Render The Final Answer

The final cell decodes the generated sequence and renders it with `IPython.display.Markdown`. Keeping `skip_special_tokens=False` is helpful in a workshop because learners can see the model's full formatted output, including any chat-template or channel tokens.

In [51]:
# Parse output
text = processor.decode(output[0], skip_special_tokens=False)
display(Markdown(text[0]))

<bos><|turn>user
What is diffusion llm?<turn|>
<|turn>model
<|channel>thought
<channel|>A **Diffusion LLM** is an emerging type of large language model that applies **diffusion processes**—the technology behind AI image generators like Midjourney and DALL-E—to text generation instead of the traditional "autoregressive" method used by GPT.

To understand Diffusion LLMs, it helps to compare them to how ChatGPT actually works today.

---

### 1. The Traditional Way: Autoregressive Models
Most current LLMs (like GPT-4 or Llama) are **autoregressive**. They generate text one token at a time, from left to right.
* **How it works:** The model looks at the prompt, predicts the next word, adds that word to the sequence, and predicts the next one.
* **The Limitation:** This is sequential. If the model makes a mistake early in a sentence, the entire following structure often drifts off because it is building on its own error.

### 2. The Diffusion Way: Diffusion LLMs
Diffusion models do not work word-by-word. Instead, they work by **refining the entire sequence at once.**
* **The Process (Forward Diffusion):** During training, the model takes clean text and gradually adds "noise" (randomness) until the text becomes completely unrecognizable gibberish.
* **The Process (Reverse Diffusion/Generation):** To generate text, the model starts with a block of "pure noise" (random tokens or mathematical representations of noise) and iteratively "denoises" the block. In each step, it adjusts the entire sentence simultaneously to make it look more like coherent text until clear words emerge.

### 3. Key Differences

| Feature | Autoregressive (GPT-style) | Diffusion LLM |
| :--- | :--- | :--- |
| **Generation Order** | One-by-one (left-to-right) | All-at-once (parallel) |
| **Speed** | Fast for short text, slow for long | Can be slow (requires many refinement steps) |
| **Global Context** | Sees only what has already been written | Sees and adjusts the whole sentence at every step |
| **Error Correction** | Can "hallucinate" and drift away | Can "fix" parts of the sentence during refinement |

### 4. Why build Diffusion LLMs?
If GPT-4 works so well, why bother with diffusion? Researchers are exploring this for several reasons:

*   **Non-Linear Editing:** Because diffusion looks at the whole sentence at once, it is potentially better at tasks that require global structure or symmetry, rather than just predicting the next likely word.
*   **Constrained Generation:** Diffusion models are often better at following strict rules (e.g., "write a poem where every third word starts with S") because they can adjust the beginning and end of the sentence to fit the constraint simultaneously.
*   **Diversity:** Diffusion models are known for their ability to sample from very complex distributions, which can lead to more "creative" or less repetitive text compared to standard models.

### 5. Current Status
Currently, **Diffusion LLMs are not the industry standard.** The primary challenge is that text is "discrete" (words are separate units), whereas diffusion works best on "continuous" data (like pixels in an image). Making diffusion work for words is mathematically difficult.

However, researchers (like those at **Stanford (Diffusion-NLP)** and others) are constantly publishing papers to bridge this gap, potentially changing how AI "thinks" about writing in the future.<eos><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>